# Sprint 1 class balance and temporal composition

This notebook computes the two pre-model diagnostics required by Sprint 1: resolved-loan class balance and quarterly loan volume/default rate. It reads only `loan_status` and `issue_d` from a configured CSV in chunks, so it can run against the full LendingClub accepted-loan file or a schema-compatible test sample.

The notebook contains no cached empirical results and never substitutes synthetic observations. When the configured file is absent, it prints setup guidance and skips analysis/output creation.

## Configuration

Set `LOAN_DATA_PATH` to override the repository default `data/raw/accepted_loans.csv`. Set `EDA_OUTPUT_DIR` to redirect generated tables, figures, and run metadata. `EDA_CHUNK_SIZE` controls streaming batch size. For an archival run, set `EDA_HASH_INPUT=1` to record the input SHA-256.

In [ ]:
from __future__ import annotations

import csv
import json
import os
from collections import Counter, defaultdict
from hashlib import sha256
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

cwd = Path.cwd().resolve()
if (cwd / 'docs').is_dir():
    REPO_ROOT = cwd
elif (cwd.parent / 'docs').is_dir():
    REPO_ROOT = cwd.parent
else:
    REPO_ROOT = cwd

data_setting = Path(os.getenv('LOAN_DATA_PATH', 'data/raw/accepted_loans.csv')).expanduser()
DATA_PATH = data_setting if data_setting.is_absolute() else REPO_ROOT / data_setting
output_setting = Path(os.getenv('EDA_OUTPUT_DIR', 'data/reports/eda')).expanduser()
OUTPUT_DIR = output_setting if output_setting.is_absolute() else REPO_ROOT / output_setting
CHUNK_SIZE = max(1, int(os.getenv('EDA_CHUNK_SIZE', '250000')))
HASH_INPUT = os.getenv('EDA_HASH_INPUT', '0').strip().lower() in {'1', 'true', 'yes'}

REQUIRED_COLUMNS = ('loan_status', 'issue_d')
TARGET_MAP = {'Charged Off': 1, 'Default': 1, 'Fully Paid': 0}
NORMALIZED_TARGET_MAP = {key.casefold(): value for key, value in TARGET_MAP.items()}
DATA_AVAILABLE = DATA_PATH.is_file()

display(pd.DataFrame([{
    'repository_root': str(REPO_ROOT),
    'input_path': str(DATA_PATH),
    'input_present': DATA_AVAILABLE,
    'output_directory': str(OUTPUT_DIR),
    'chunk_size': CHUNK_SIZE,
    'hash_input': HASH_INPUT,
}]))

if not DATA_AVAILABLE:
    display(Markdown(
        '**Analysis skipped:** no configured CSV exists. Set `LOAN_DATA_PATH` to a '
        'LendingClub accepted-loan CSV or a schema-compatible sample; no synthetic '
        'or placeholder findings were generated.'
    ))

## Streaming aggregation

Only three statuses are resolved for this study. All other statuses, including current or grace-period loans, remain visible in the status inventory but are excluded from the binary target. Quarterly rates use resolved loans with a valid `issue_d` only.

In [ ]:
def file_sha256(path: Path, block_size: int = 1024 * 1024) -> str:
    digest = sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(block_size), b''):
            digest.update(block)
    return digest.hexdigest()


def find_header_row(path: Path, scan_limit: int = 100) -> tuple[int, list[str]]:
    with path.open('r', encoding='utf-8-sig', errors='replace', newline='') as stream:
        for line_number, line in enumerate(stream):
            if line_number >= scan_limit:
                break
            fields = [field.strip() for field in next(csv.reader([line]))]
            if set(REQUIRED_COLUMNS).issubset(fields):
                return line_number, fields
    raise ValueError(
        f'Could not find a CSV header containing {REQUIRED_COLUMNS} in the first '
        f'{scan_limit} lines of {path}'
    )


def parse_issue_month(values: pd.Series) -> pd.Series:
    cleaned = values.astype('string').str.strip()
    parsed = pd.to_datetime(cleaned, format='%b-%Y', errors='coerce')
    fallback_mask = parsed.isna() & cleaned.notna()
    if fallback_mask.any():
        parsed.loc[fallback_mask] = pd.to_datetime(
            cleaned.loc[fallback_mask], errors='coerce'
        )
    return parsed


def scan_loan_csv(path: Path, chunk_size: int) -> dict[str, Any]:
    header_row, header_columns = find_header_row(path)
    duplicate_columns = sorted({
        column for column in header_columns if header_columns.count(column) > 1
    })
    if duplicate_columns:
        raise ValueError(f'Duplicate CSV columns are not allowed: {duplicate_columns}')
    missing_columns = sorted(set(REQUIRED_COLUMNS) - set(header_columns))
    if missing_columns:
        raise ValueError(
            f'Missing required columns {missing_columns}; available columns include '
            f'{header_columns[:20]}'
        )

    status_counts: Counter[str] = Counter()
    class_counts: Counter[int] = Counter()
    quarter_counts: dict[str, list[int]] = defaultdict(lambda: [0, 0])
    total_rows = 0
    invalid_issue_date_resolved_rows = 0

    reader = pd.read_csv(
        path,
        skiprows=header_row,
        usecols=list(REQUIRED_COLUMNS),
        chunksize=chunk_size,
        low_memory=False,
    )
    for chunk in reader:
        total_rows += len(chunk)
        status = chunk['loan_status'].astype('string').str.strip()
        status_for_count = status.fillna('<MISSING>').replace('', '<MISSING>')
        status_counts.update(status_for_count.tolist())

        target = status.str.casefold().map(NORMALIZED_TARGET_MAP)
        resolved_mask = target.notna()
        class_counts.update(target.loc[resolved_mask].astype('int8').tolist())

        issue_month = parse_issue_month(chunk['issue_d'])
        temporal_mask = resolved_mask & issue_month.notna()
        invalid_issue_date_resolved_rows += int((resolved_mask & issue_month.isna()).sum())

        temporal = pd.DataFrame({
            'target': target.loc[temporal_mask].astype('int8'),
            'issue_quarter': issue_month.loc[temporal_mask].dt.to_period('Q').astype(str),
        })
        if not temporal.empty:
            grouped = temporal.groupby('issue_quarter', sort=False)['target'].agg(
                resolved_loans='size', defaults='sum'
            )
            for quarter, row in grouped.iterrows():
                quarter_counts[str(quarter)][0] += int(row['resolved_loans'])
                quarter_counts[str(quarter)][1] += int(row['defaults'])

    resolved_rows = int(sum(class_counts.values()))
    return {
        'total_rows': int(total_rows),
        'resolved_rows': resolved_rows,
        'excluded_unresolved_or_unmapped_rows': int(total_rows - resolved_rows),
        'invalid_issue_date_resolved_rows': int(invalid_issue_date_resolved_rows),
        'temporal_rows': int(sum(values[0] for values in quarter_counts.values())),
        'status_counts': dict(sorted(status_counts.items())),
        'class_counts': dict(class_counts),
        'quarter_counts': dict(quarter_counts),
    }

In [ ]:
scan_result: dict[str, Any] | None = None
run_metadata: dict[str, Any] | None = None

if DATA_AVAILABLE:
    scan_result = scan_loan_csv(DATA_PATH, CHUNK_SIZE)
    if scan_result['resolved_rows'] == 0:
        raise ValueError('No resolved loans matched the documented target mapping.')
    if scan_result['temporal_rows'] == 0:
        raise ValueError('No resolved loans had a valid issue_d for quarterly analysis.')

    input_stat = DATA_PATH.stat()
    run_metadata = {
        'input_path': str(DATA_PATH.resolve()),
        'input_size_bytes': int(input_stat.st_size),
        'input_sha256': file_sha256(DATA_PATH) if HASH_INPUT else None,
        'sha256_capture_note': (
            'computed'
            if HASH_INPUT
            else 'not computed; rerun with EDA_HASH_INPUT=1 for archival output'
        ),
        'required_columns': list(REQUIRED_COLUMNS),
        'target_mapping': TARGET_MAP,
        'chunk_size': CHUNK_SIZE,
        **{key: value for key, value in scan_result.items() if key != 'quarter_counts'},
    }
    display(pd.DataFrame([{
        'total_rows': scan_result['total_rows'],
        'resolved_rows': scan_result['resolved_rows'],
        'excluded_unresolved_or_unmapped_rows': scan_result['excluded_unresolved_or_unmapped_rows'],
        'resolved_rows_missing_or_invalid_issue_d': scan_result['invalid_issue_date_resolved_rows'],
        'resolved_rows_in_temporal_analysis': scan_result['temporal_rows'],
    }]))

## Resolved-loan class balance

The denominator is the set of rows mapped by the documented resolved-status rule, before any train/validation/test split. The table and chart below are data-generated when an input is configured.

In [ ]:
class_balance: pd.DataFrame | None = None
class_balance_figure = None

if scan_result is not None:
    class_balance = pd.DataFrame([
        {'target': 0, 'class_label': 'Fully Paid', 'count': scan_result['class_counts'].get(0, 0)},
        {
            'target': 1,
            'class_label': 'Default or Charged Off',
            'count': scan_result['class_counts'].get(1, 0),
        },
    ])
    class_balance['share'] = class_balance['count'] / class_balance['count'].sum()
    class_balance_display = class_balance.copy()
    class_balance_display['share'] = class_balance_display['share'].map('{:.2%}'.format)
    display(class_balance_display)

    class_balance_figure, axis = plt.subplots(figsize=(7, 4))
    bars = axis.bar(
        class_balance['class_label'],
        class_balance['count'],
        color=['#4C78A8', '#E45756'],
    )
    axis.set_title('Resolved-loan class balance')
    axis.set_ylabel('Loan count')
    axis.ticklabel_format(axis='y', style='plain')
    axis.bar_label(bars, labels=[f"{share:.1%}" for share in class_balance['share']], padding=3)
    class_balance_figure.tight_layout()
    plt.show()

## Quarterly temporal composition

Volume is the number of resolved loans with a valid issue quarter. Default rate is defaults divided by resolved loans within that quarter. This notebook does not pre-assert a direction or magnitude of temporal change; interpretation must use the generated table and plots.

In [ ]:
temporal_composition: pd.DataFrame | None = None
temporal_figure = None

if scan_result is not None:
    temporal_composition = pd.DataFrame([
        {
            'issue_quarter': quarter,
            'resolved_loans': values[0],
            'defaults': values[1],
        }
        for quarter, values in scan_result['quarter_counts'].items()
    ])
    temporal_composition['_quarter_order'] = pd.PeriodIndex(
        temporal_composition['issue_quarter'], freq='Q'
    )
    temporal_composition = temporal_composition.sort_values('_quarter_order').drop(
        columns='_quarter_order'
    ).reset_index(drop=True)
    temporal_composition['default_rate'] = (
        temporal_composition['defaults'] / temporal_composition['resolved_loans']
    )
    temporal_display = temporal_composition.copy()
    temporal_display['default_rate'] = temporal_display['default_rate'].map(
        '{:.2%}'.format
    )
    display(temporal_display)

    temporal_figure, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
    axes[0].bar(
        temporal_composition['issue_quarter'],
        temporal_composition['resolved_loans'],
        color='#4C78A8',
    )
    axes[0].set_title('Resolved-loan volume by issue quarter')
    axes[0].set_ylabel('Loan count')
    axes[0].ticklabel_format(axis='y', style='plain')

    axes[1].plot(
        temporal_composition['issue_quarter'],
        temporal_composition['default_rate'],
        color='#E45756',
        marker='o',
        linewidth=1.5,
        markersize=3,
    )
    axes[1].set_title('Resolved-loan default rate by issue quarter')
    axes[1].set_ylabel('Default rate')
    axes[1].set_xlabel('Issue quarter')
    axes[1].yaxis.set_major_formatter(lambda value, position: f'{value:.0%}')
    tick_step = max(1, len(temporal_composition) // 12)
    for index, label in enumerate(axes[1].get_xticklabels()):
        label.set_visible(index % tick_step == 0)
        label.set_rotation(45)
        label.set_ha('right')
    temporal_figure.tight_layout()
    plt.show()

## Reproducible outputs

When data is available, this cell writes the aggregate tables, figures, and run metadata. These outputs contain no row-level loan data. Archive them with the acquisition manifest, source digest, environment lock, and code revision used for the run.

In [ ]:
if scan_result is not None:
    assert class_balance is not None
    assert temporal_composition is not None
    assert run_metadata is not None
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    class_balance.to_csv(OUTPUT_DIR / 'class_balance.csv', index=False)
    temporal_composition.to_csv(OUTPUT_DIR / 'temporal_composition_by_quarter.csv', index=False)
    if class_balance_figure is not None:
        class_balance_figure.savefig(OUTPUT_DIR / 'class_balance.png', dpi=160, bbox_inches='tight')
    if temporal_figure is not None:
        temporal_figure.savefig(
            OUTPUT_DIR / 'temporal_composition.png',
            dpi=160,
            bbox_inches='tight',
        )
    with (OUTPUT_DIR / 'eda_run_metadata.json').open('w', encoding='utf-8') as stream:
        json.dump(run_metadata, stream, indent=2, sort_keys=True)
        stream.write('\n')

    display(Markdown(
        f'Wrote aggregate EDA artifacts to `{OUTPUT_DIR}`. No row-level records were exported.'
    ))